# N037 · 基础排序与比较模型

**目标：** 手写简单排序并通过反例理解稳定性。

**先修：** N017, N011。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 插入/选择/冒泡；比较与交换；稳定性；原地；排序后置条件。

**配套讲解来源：** [同名逐章意见](../../comment/037_elementary_sorts.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

表面问题很简单：**把一个数组从小到大排好**。输入 `[4,2,3,1]`，输出 `[1,2,3,4]`——这是 “运行示例”部分 用的真实数据。但这一章真正的目标不是背三段排序代码，而是三件事：第一，亲手写出三种最经典的简单排序（插入、选择、冒泡），并说出每一种"每一轮到底完成了什么"；第二，体会"比较排序"这个模型——只通过两两比较键的大小来决定顺序；第三，通过一个具体反例搞懂**稳定性**：排序之后，键相等的元素是否还保持它们原来的先后顺序。

稳定性看一个具体例子（“自动测试”部分 的真实数据）：输入四条记录 `[(2,'a'),(1,'b'),(2,'c'),(1,'d')]`，按第一个数字（键）排序，稳定排序的期望输出是 `[(1,'b'),(1,'d'),(2,'a'),(2,'c')]`。为什么 (1,'b') 必须排在 (1,'d') 前面？因为排序前 'b' 那条记录就在 'd' 前面，而两者的键都是 1，稳定排序不许改变相等键元素的相对位置；同理 (2,'a') 必须在 (2,'c') 前面。如果哪个排序把这两对翻了个个儿，它就是不稳定的。

## 2. 基础知识：先读懂这些词

- **比较排序（comparison sort）**：只通过"比较两个元素的键谁大谁小"来决定排列顺序的排序。插入、选择、冒泡、归并、快排、堆排都属于这一族；它们都要面对 Ω(n log n) 的最坏情形下界。
- **插入排序（insertion sort）**：像整理手中的扑克牌，维护"左边一段已经排好"的前缀，每轮把下一个新元素插进前缀里合适的位置。
- **选择排序（selection sort）**：每轮从未排序区域里选出最小的元素，把它换到未排序区域的开头，于是排序区域从左往右一格格变长。
- **冒泡排序（bubble sort）**：每轮从头到尾比较相邻元素并交换逆序对，让当前最大的元素一步步"冒"到未排序区的末尾。
- **循环不变量（每轮已完成的部分）**：“思路推导”部分 说"排序的第一步不是背交换语句，而是定义每轮已完成的部分"。三种排序分别维护的不变量是：插入——前缀有序；选择——前缀有序且是全数组最小的若干个；冒泡——后缀有序且是全数组最大的若干个。
- **稳定性（stability）**：键相等的元素在输出中保持输入里的先后顺序。实现上的关键是"只在严格大于时才移动/交换"——相等就不动，后来者永远不越过相等的先行者。
- **原地排序（in-place）**：只在原数组上移动元素、只花 O(1) 的额外空间（几个变量），不再开一条同长度的辅助数组。
- **key 参数（`key=lambda x:x`）**：告诉排序"按元素的哪个属性来比"。默认按元素本身比；测试里传 `key=lambda x:x[0]` 表示按元组第一个字段比。
- **提前终止（冒泡的 changed 标志）**：冒泡某一轮一次交换都没发生，说明数组已经有序，可以立刻收工，这让冒泡在已排序输入上只花 O(n)。
- **排序后置条件**：程序跑完后必须满足的两件事——输出是输入的一个重排（元素没丢没多），且输出非降。测试就拿这两条对照 `sorted(a)`。

## 3. 思路推导：从小例子开始

- **Step 1（插入）**：把数组看成"有序前缀 + 未排序后缀"。取后缀第一个元素 value，在前缀里从右往左比较，凡是严格大于 value 的元素都右移一格，腾出的空位放进 value。前缀长度加一，仍然有序。
- **Step 2（选择）**：未排序区从位置 i 开始，用一根扫描指针找出其中键最小的位置 best，然后交换 a[i] 与 a[best]。位置 i 归入排序区。
- **Step 3（冒泡）**：未排序区是 a[0..end)，相邻两两比较，前者键严格大于后者就交换；一轮下来最大值到达 a[end-1]，end 减一。若整轮 changed 为 False，提前结束。

**手算演示一：插入排序 `[4,2,3,1]`（“运行示例”部分 表格的数据，逐轮对照）。**

| 阶段 | 数组 |
|---|---|
| 初始 | [4,2,3,1] |
| 插入第1项（值 2） | [2,4,3,1] |
| 插入第2项（值 3） | [2,3,4,1] |
| 插入第3项（值 1） | [1,2,3,4] |

第 1 项是 2：前缀里 4 严格大于 2，右移，2 插在 4 前面。第 2 项是 3：4 大于 3 右移，2 不大于 3 停住，3 插在 2 和 4 之间。第 3 项是 1：4、3、2 全都严格大于 1，全部右移，1 落到最前面。

**手算演示二：冒泡排序 `[3,1,2]`。** end=2 这轮：比较 (3,1) 逆序，交换得 [1,3,2]；比较 (3,2) 逆序，交换得 [1,2,3]；changed=True。end=1 这轮：比较 (1,2) 不逆序，整轮无交换，changed=False，提前结束。总共两轮就收工。

**手算演示三：选择排序的不稳定反例 `[(2,'a'),(2,'b'),(1,'x')]`（键取首字段）。** 第 0 轮在三条记录里找最小键，找到 (1,'x')，与位置 0 的 (2,'a') 交换，数组变成 `[(1,'x'),(2,'b'),(2,'a')]`。注意：(2,'a') 和 (2,'b') 键相等、原来 'a' 在 'b' 前，这次远距离交换之后 'b' 跑到了 'a' 前面——相对顺序被翻掉了，这就是"不稳定"的实例。这个例子也是练习 2 的核心素材。

## 4. 核心代码：insertion_sort

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def insertion_sort(a, key=lambda x: x):
    for i in range(1, len(a)):
        value = a[i]
        j = i
        while j > 0 and key(a[j - 1]) > key(value):
            a[j] = a[j - 1]
            j -= 1
        a[j] = value
    return a
```

### 逐段读懂代码

### 函数 1：`insertion_sort` —— 插入排序

```python
def insertion_sort(a, key=lambda x: x):
    for i in range(1, len(a)):
        value = a[i]
        j = i
        while j > 0 and key(a[j - 1]) > key(value):
            a[j] = a[j - 1]
            j -= 1
        a[j] = value
    return a
```

- `key=lambda x:x`：默认按元素本身比较；调用方可以传别的 lambda 换比较依据（“自动测试”部分 的稳定性测试就传了 `key=lambda x:x[0]`）。
- `for i in range(1,len(a))`：从第二个元素开始插，因为单个元素的前缀天然有序，i=0 不用处理。
- `value=a[i]; j=i`：先把待插元素存进 value（待会儿右移会覆盖 a[i]），j 表示当前空位的位置。
- `while j>0 and key(a[j-1])>key(value):`：注意两个细节。其一，`j>0` 是边界守卫，防止空位已经顶到最左边还去取 a[-1]（Python 的负索引会悄悄取到末尾，不报错但结果是错的，所以必须显式挡住）。其二，比较用的是严格大于 `>`：前一个元素与 value 相等时不停右移而是就地停下——这正是插入排序稳定的原因。
- `a[j]=a[j-1]; j-=1`：把前一个元素右移一格，空位左移一格。
- `a[j]=value`：循环停下时 j 就是 value 的落位处。
- `return a`：原地排完后把同一个列表返回，方便链式断言。

### 函数 2：`selection_sort` —— 选择排序

```python
def selection_sort(a, key=lambda x: x):
    for i in range(len(a)):
        best = i
        for j in range(i + 1, len(a)):
            if key(a[j]) < key(a[best]):
                best = j
        a[i], a[best] = (a[best], a[i])
    return a
```

- `for i in range(len(a))`：i 是"排序区与未排序区的分界线"，逐轮右移。
- `best=i`：先假设分界线上的元素就是未排序区最小的。
- `for j in range(i+1,len(a)): if key(a[j])<key(a[best]): best=j`：扫一遍未排序区，键严格小于当前最小才更新 best。用严格小于意味着并列时保留"最早出现"的那个，但这个细节救不了稳定性——见下一行。
- `a[i],a[best]=a[best],a[i]`：把最小元素换到分界线上。这是一次**远距离交换**：它可能跨过若干与 a[i] 键相等的元素，把它们的相对顺序翻掉（第三节手算演示三就是现场翻车实录），所以选择排序通常不稳定。best==i 时交换是自己换自己，无害。
- `return a`：同样原地返回。

### 函数 3：`bubble_sort` —— 冒泡排序

```python
def bubble_sort(a, key=lambda x: x):
    for end in range(len(a) - 1, 0, -1):
        changed = False
        for i in range(end):
            if key(a[i]) > key(a[i + 1]):
                a[i], a[i + 1] = (a[i + 1], a[i])
                changed = True
        if not changed:
            break
    return a
```

- `for end in range(len(a)-1,0,-1)`：end 是本轮要冒泡到达的终点（不含），从最后一个位置倒着往前收。每完成一轮，a[end..] 就固定为最大的若干元素且有序。
- `changed=False`：每轮开始清零，用来统计本轮是否发生过交换。
- `if key(a[i])>key(a[i+1]):`：相邻比较，仍然用严格大于——相等就不交换，所以冒泡稳定，相等的元素永远保持原顺序。
- `a[i],a[i+1]=a[i+1],a[i]; changed=True`：交换相邻逆序对并记账。
- `if not changed: break`：整轮一次交换都没有，说明数组已经整体有序，立即退出。这是冒泡在已排序输入上只花 O(n) 的原因（练习 1 会继续压缩无效扫描）。
- `return a`：原地返回。

## 5. 分段实现：按顺序运行

**本章接口：** `insertion_sort(a)`、`selection_sort(a)`、`bubble_sort(a)`

### insertion_sort

In [1]:
def insertion_sort(a, key=lambda x: x):
    for i in range(1, len(a)):
        value = a[i]
        j = i
        while j > 0 and key(a[j - 1]) > key(value):
            a[j] = a[j - 1]
            j -= 1
        a[j] = value
    return a

### selection_sort

In [2]:
def selection_sort(a, key=lambda x: x):
    for i in range(len(a)):
        best = i
        for j in range(i + 1, len(a)):
            if key(a[j]) < key(a[best]):
                best = j
        a[i], a[best] = (a[best], a[i])
    return a

### bubble_sort

In [3]:
def bubble_sort(a, key=lambda x: x):
    for end in range(len(a) - 1, 0, -1):
        changed = False
        for i in range(end):
            if key(a[i]) > key(a[i + 1]):
                a[i], a[i + 1] = (a[i + 1], a[i])
                changed = True
        if not changed:
            break
    return a

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[4,2,3,1]; rows=[('初始',a[:])]
for i in range(1,len(a)):
    value=a[i]; j=i
    while j>0 and a[j-1]>value: a[j]=a[j-1]; j-=1
    a[j]=value; rows.append((f'插入第{i}项',a[:]))
show_table(['阶段','数组'],rows)

阶段,数组
初始,"[4, 2, 3, 1]"
插入第1项,"[2, 4, 3, 1]"
插入第2项,"[2, 3, 4, 1]"
插入第3项,"[1, 2, 3, 4]"


## 7. 正确性、适用条件与复杂度

每轮保持已排序区域不变量并扩大一区域；最终覆盖全数组。选择排序交换远距离元素，通常不稳定；不要因为样例未暴露就称其稳定。

**代价：** 最坏均 O(n²)，辅助空间 O(1)；插入与优化冒泡在已排序输入上 O(n)。比较排序一般下界为 Ω(n log n)，是对所有输入的最坏情形而言。

### 展开理解

**插入排序为什么对？** 我们把论证说成完整的话。开头：第一轮开始前，前缀只有 a[0] 一个元素，单个元素当然有序。中间：假设某一轮开始前缀 a[0..i-1] 有序，内层 while 把所有严格大于 value 的元素依次右移，停在第一个不大于 value 的位置，把 value 放进去之后，前缀变成 a[0..i] 且仍然有序——每轮做完，"前缀有序"这条性质都完好如初。结尾：i 从 1 走到 n-1，循环有限轮必停；最后一轮结束，前缀就是整个数组，于是整个数组有序。

**选择排序为什么对？** 每一轮，我们在未排序区里选出的确是最小的元素（内层循环把每个候选都比过了一遍），把它放到分界线 i 上之后，a[0..i] 是全数组最小的 i+1 个元素并且排好了顺序。i 走到头，整个数组有序。但请注意它**不保证稳定**：“正确性与复杂度”部分 特别警告"不要因为样例未暴露就称其稳定"——第三节那个 3 元素反例就是证据，稳定性要么证明、要么举反例，不能靠跑几个样例碰运气。

**冒泡排序为什么对？** 每一轮的相邻比较与交换，保证未排序区里最大的元素被一路推到本轮终点；于是每完成一轮，数组尾部就多固化一个"当前剩余中的最大值"。end 从右往左收，收完整个数组就是非降的。changed 标志只影响"什么时候停"，不影响排出来的结果。

**复杂度是多少？** 三种排序最坏都是 O(n²)：插入是 1+2+…+(n-1) 次比较，选择是不管输入好坏固定约 n²/2 次比较，冒泡最坏也是约 n²/2 次比较。拿具体数字感受：n=1000 时大约 50 万到 100 万次比较，一瞬间；n=10⁵ 时就是百亿量级，等不起。辅助空间三边都是 O(1)（原地，只用了 i、j、value、best、changed 这几个变量）。两个好消息：插入排序和带提前终止的冒泡在"输入已经有序"时只花 O(n)——插入每轮 while 立即失败、冒泡第一轮就 break。另外，“正确性与复杂度”部分 提醒比较排序有 Ω(n log n) 的最坏情形下界：任何靠两两比较的排序，总存在让它们吃满约 n log n 次比较的输入，n=10⁵ 时约 170 万次——这正是后面三章（归并、快排、堆）要兑现的量级。

三种排序一张表看清差异：

| 排序 | 每轮结束时保证什么 | 稳定吗 | 最好情形 | 最坏情形 |
|---|---|---|---|---|
| 插入 | 前缀 a[0..i] 有序 | 稳定（严格大于才移动） | O(n)（已有序） | O(n²) |
| 选择 | a[0..i] 是最小 i+1 个且有序 | 不稳定（远距离交换翻序） | O(n²)（照扫不误） | O(n²) |
| 冒泡 | a[end..] 是最大若干且有序 | 稳定（相等不交换） | O(n)（一轮无交换即停） | O(n²) |

值得对照的一行是"最好情形"：插入排序在"基本有序"的输入上极快（每个新元素只比一两下就落位），这个性质让它在工程上常被用作小段或近有序段的收尾排序；选择排序则无论输入好坏都扫满，唯一的优点是交换次数最多 n−1 次。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言分两块：

- **穷举正确性块**：`for n in range(7): for a in product([-1,0,1],repeat=n): for fn in [insertion_sort,selection_sort,bubble_sort]: assert fn(list(a))==sorted(a)`。它枚举长度 0 到 6、取值只有 −1/0/1 的**所有**数组（共 1+3+9+…+729 = 1093 个），对每个数组把三种排序的结果与内置 `sorted` 逐一比对。取值压到三种是为了制造大量相等元素，专治比较条件写错（比如把 `>` 写成 `>=`）这类在"全不相等"数据上看不出来的 bug。注意每次都传 `list(a)` 复制一份，因为三个函数都是原地修改，不复制会把 product 生成的元组……准确说是防止排序结果互相污染调用现场。
- **稳定性断言块**：`a=[(2,'a'),(1,'b'),(2,'c'),(1,'d')]`，对 insertion_sort 和 bubble_sort 传 `key=lambda x:x[0]`，断言结果恰为 `[(1,'b'),(1,'d'),(2,'a'),(2,'c')]`。这条测的是**相等键的先后保持**：两个键为 1 的记录里 'b' 必须仍在 'd' 前，两个键为 2 的记录里 'a' 必须仍在 'c' 前。为什么 selection_sort 不参加这条测试？因为它通不过——第三节反例已经展示它会翻掉相等元素的顺序。测试前先 `a[:]` 复制，防止原地排序破坏原数组导致后续断言用到脏数据。

两块断言的分工一览：

| 断言块 | 类型 | 它防的是哪类 bug |
|---|---|---|
| 穷举 1093 个 −1/0/1 数组对拍 sorted | 穷举交叉验证 | 比较条件写错（如 > 写成 >=） |
| 插入/冒泡按 key 排元组 | 稳定性 | 相等键元素被翻序 |

为什么穷举要到长度 6？长度 6、取值 3 种共 729 个数组，足以让"相等元素相邻、隔位、成片"各种形态都出现；而三层循环总量仍在千级、一秒内跑完——穷举测试的甜点区就是"小而全"。

In [5]:
from itertools import product

for n in range(7):
    for a in product([-1, 0, 1], repeat=n):
        for fn in [insertion_sort, selection_sort, bubble_sort]:
            assert fn(list(a)) == sorted(a)

a = [(2, 'a'), (1, 'b'), (2, 'c'), (1, 'd')]

for fn in [insertion_sort, bubble_sort]:
    assert fn(a[:], key=lambda x: x[0]) == [(1, 'b'), (1, 'd'), (2, 'a'), (2, 'c')]

print('N037: 所有本章断言通过')

N037: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 减少冒泡无效扫描。

**练习 2：** 寻找选择排序破坏稳定性的最小例子。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（减少冒泡的无效扫描）**：方向提示——冒泡一轮结束后，"最后一次交换发生的位置"之后的段落其实已经有序，下一轮的 end 可以直接设成那个位置，而不必一格一格减。更进一步的版本是双向冒泡（鸡尾酒排序），一轮正向把最大推到右边、一轮反向把最小推到左边，对付"小元素卡在末尾"的输入特别有效。手算示例建议用 [1,3,2,4,5]：普通版第二轮还要扫到底，优化版第二轮 end 可以直接跳到 2 附近就收工。验证时对拍 `sorted(a)` 即可。
- **练习 2（寻找选择排序破坏稳定性的最小例子）**：方向提示——要让不稳定显形，需要"两个键相等的元素"中间隔着一个"键更小的元素"，这样最小元素被换到前面时才会跨越并翻掉那对相等元素。第三节已给出 3 元素反例 `[(2,'a'),(2,'b'),(1,'x')]`（键取首字段）。你可以再论证一下 2 个元素为什么不够：两个元素里放不下"一对相等键加一个更小键"三件事，所以 3 是最小规模。写清输入、期望输出（稳定版应得 `[(1,'x'),(2,'a'),(2,'b')]`）与实际输出，再用 selection_sort 跑一遍验证。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
def insertion_sort(a, key=lambda x: x):
    for i in range(1, len(a)):
        value = a[i]
        j = i
        while j > 0 and key(a[j - 1]) > key(value):
            a[j] = a[j - 1]
            j -= 1
        a[j] = value
    return a

def selection_sort(a, key=lambda x: x):
    for i in range(len(a)):
        best = i
        for j in range(i + 1, len(a)):
            if key(a[j]) < key(a[best]):
                best = j
        a[i], a[best] = (a[best], a[i])
    return a

def bubble_sort(a, key=lambda x: x):
    for end in range(len(a) - 1, 0, -1):
        changed = False
        for i in range(end):
            if key(a[i]) > key(a[i + 1]):
                a[i], a[i + 1] = (a[i + 1], a[i])
                changed = True
        if not changed:
            break
    return a

# 示例与回归测试
from itertools import product

for n in range(7):
    for a in product([-1, 0, 1], repeat=n):
        for fn in [insertion_sort, selection_sort, bubble_sort]:
            assert fn(list(a)) == sorted(a)

a = [(2, 'a'), (1, 'b'), (2, 'c'), (1, 'd')]

for fn in [insertion_sort, bubble_sort]:
    assert fn(a[:], key=lambda x: x[0]) == [(1, 'b'), (1, 'd'), (2, 'a'), (2, 'c')]

print('N037: 所有本章断言通过')

N037: 所有本章断言通过


## 11. 代表题与迁移

“题目映射”部分 列出的题目：

- **912. Sort an Array（排序数组）**：给你一个整数数组，要求排序后返回。这题练的是"能不能独立写出一个能通过大规模数据的排序"。本章三个 O(n²) 排序在 LeetCode 的 n=5×10⁴ 规模下会超时，它们是练手地基；后面 N038（归并）、N039（快排）、N040（堆排）会给出能过的 O(n log n) 版本，到时候拿同一题回头对比。

照 notebook 原文提醒：以上条目用于知识映射，本章实现遵循教学契约，刷题时以 LeetCode 官方题面与评测为准。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。